In [1]:
#here we are going to build a agentic rag system 
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq 
from dotenv import load_dotenv 
load_dotenv() 
from langchain_classic.tools import tool
import os
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings

c:\Users\aashi\Envs\lang_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm=ChatGroq(model='llama-3.1-8b-instant') 

In [3]:
llm.invoke('hi')

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.008280266, 'completion_tokens_details': None, 'prompt_time': 0.002246926, 'prompt_tokens_details': None, 'queue_time': 0.052323383, 'total_time': 0.010527192}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0bd7-9ed0-73a1-abdd-8fb15a3eed92-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})

In [4]:
#now lets build a vector db 
pdf=PyPDFLoader(file_path='./data/John_F_Kennedy.pdf')

In [5]:
docs=pdf.load()

In [6]:
#each page is seperate chunk 
print(len(docs))

34


In [7]:
#now lets do the splitting in it 
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

In [8]:
chunks=text_splitter.split_documents(docs)

In [9]:
len(chunks)

142

In [ ]:
#now lets store it in the vector db 
#NOTE: DO this for 1 time and then store the vector db in disk
from langchain_classic.vectorstores import FAISS
embedding_model=CohereEmbeddings(model='embed-multilingual-light-v3.0')
vectordb=FAISS.from_documents(documents=chunks,embedding=embedding_model)


TypeError: FAISS.save_local() missing 1 required positional argument: 'folder_path'

In [13]:
vectordb.save_local(folder_path='vector_db',index_name='john_f_kennedy_vector_db')

In [ ]:
vectordb = FAISS.load_local(folder_path='vector_db',index_name='john_f_kennedy_vector_db',allow_dangerous_deserialization=True,embeddings=embedding_model)

In [ ]:
vectordb.similarity_search(k=3,query='who is john f kennedy')

[Document(id='2a725aa4-007c-49cd-b2db-6933e8cc3108', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'needappearances': 'True', 'source': './data/John_F_Kennedy.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content="John F. Kennedy\nOval Office portrait, 1963\n35th President of the United States\nIn office\nJanuary 20, 1961 – November 22, 1963\nVice President Lyndon B. Johnson\nPreceded by Dwight D. Eisenhower\nSucceeded by Lyndon B. Johnson\nUnited States Senator\nfrom Massachusetts\nIn office\nJanuary 3, 1953 – December 22, 1960\nPreceded by Henry Cabot Lodge Jr.\nSucceeded by Benjamin A. Smith II\nMember of the U.S. House of Representatives\nfrom Massachusetts's 11th district\nIn office\nJanuary 3, 1947 – January 3, 1953\nPreceded by James Michael Curley\nSucceeded by Tip O'Neill\nPersonal details\nBorn John Fitzgerald Kennedy\nMay 29, 1917\nBrookline, Massachusetts,\nU.S.\nJohn F. Kennedy\nJohn Fitzgerald Kennedy (May 29, 1917 –\nNovember 22, 

In [ ]:
# vectordb.similarity_search(k=3,query='who is john f kennedy')

[Document(id='2a725aa4-007c-49cd-b2db-6933e8cc3108', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'needappearances': 'True', 'source': './data/John_F_Kennedy.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content="John F. Kennedy\nOval Office portrait, 1963\n35th President of the United States\nIn office\nJanuary 20, 1961 – November 22, 1963\nVice President Lyndon B. Johnson\nPreceded by Dwight D. Eisenhower\nSucceeded by Lyndon B. Johnson\nUnited States Senator\nfrom Massachusetts\nIn office\nJanuary 3, 1953 – December 22, 1960\nPreceded by Henry Cabot Lodge Jr.\nSucceeded by Benjamin A. Smith II\nMember of the U.S. House of Representatives\nfrom Massachusetts's 11th district\nIn office\nJanuary 3, 1947 – January 3, 1953\nPreceded by James Michael Curley\nSucceeded by Tip O'Neill\nPersonal details\nBorn John Fitzgerald Kennedy\nMay 29, 1917\nBrookline, Massachusetts,\nU.S.\nJohn F. Kennedy\nJohn Fitzgerald Kennedy (May 29, 1917 –\nNovember 22, 

In [34]:
retriever=vectordb.as_retriever(search_kwargs={'k':3})

In [35]:
retriever.invoke('who was john f kennedy')

[Document(id='4979583b-b752-43c2-af89-33aa7decfafa', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'needappearances': 'True', 'source': './data/John_F_Kennedy.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content="John F. Kennedy\nOval Office portrait, 1963\n35th President of the United States\nIn office\nJanuary 20, 1961 – November 22, 1963\nVice President Lyndon B. Johnson\nPreceded by Dwight D. Eisenhower\nSucceeded by Lyndon B. Johnson\nUnited States Senator\nfrom Massachusetts\nIn office\nJanuary 3, 1953 – December 22, 1960\nPreceded by Henry Cabot Lodge Jr.\nSucceeded by Benjamin A. Smith II\nMember of the U.S. House of Representatives\nfrom Massachusetts's 11th district\nIn office\nJanuary 3, 1947 – January 3, 1953\nPreceded by James Michael Curley\nSucceeded by Tip O'Neill\nPersonal details\nBorn John Fitzgerald Kennedy\nMay 29, 1917\nBrookline, Massachusetts,\nU.S.\nJohn F. Kennedy\nJohn Fitzgerald Kennedy (May 29, 1917 –\nNovember 22, 

In [36]:
#now lets make an ai agent with this as a tool in langchain 
@tool
def agent_rag_tool(query: str): 
    """If user ask anything about the john f kennedy then you have to call this tool"""
    docs=retriever.invoke(query) 
    return "\n".join([doc.page_content for doc in docs])


In [37]:
#prompt
prompt="""Answer the following questions as best you can. You have access to the following tools:

{tools}


Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}] if need 
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""" 

# This above is the actual prompt that we have pulled from the langchain_hub and we use it in our ReAct agent 

In [38]:
# #now tools for the general answer 
# def general_query(query: str):
#     llm.invoke()

In [39]:
from langchain_classic.prompts import PromptTemplate
template =PromptTemplate.from_template(template=prompt)

In [41]:
#check with the google models 
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
llm2=ChatGroq(model='llama-3.3-70b-versatile',temperature=0)
# llm.invoke('hi')

In [42]:
from langchain_classic.tools import Tool
from langchain_classic import hub
from langchain_classic.agents import create_react_agent,AgentExecutor
# agent_rag_tool=Tool(name="RAG Tool",func=rag_tool,description="If user ask anything about the john f kennedy then you have to call this tool")
prompt = hub.pull("hwchase17/react") #this is the same prompt that we have define above
print(prompt)
our_agent=create_react_agent(llm=llm2,prompt=template,tools=[agent_rag_tool])


agent_executor=AgentExecutor(agent=our_agent,tools=[agent_rag_tool],verbose=True,handle_parsing_errors=True)

c:\Users\aashi\Envs\lang_env\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


c:\Users\aashi\Envs\lang_env\Lib\site-packages\langsmith\client.py:7195: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


In [43]:
query='Compare the early childhood education of John F. Kennedy with the education he received at Harvard. Were they similar?'
# Now run the executor
output=agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
To answer this question, I need to know more about John F. Kennedy's early childhood education and his education at Harvard. Since the question is about John F. Kennedy, I should use the agent_rag_tool to find relevant information.

Action: agent_rag_tool
Action Input: John F. Kennedy early childhood education and Harvard educationof the United States from 1961 until his assassination
in 1963. He was the youngest person elected
president.[2] Kennedy served at the height of the Cold
War, and the majority of his foreign policy concerned
relations with the Soviet Union and Cuba. A
Democrat, Kennedy represented Massachusetts in
both houses of the U.S. Congress prior to his
presidency.
Born into the prominent Kennedy family in Brookline,
Massachusetts, Kennedy graduated from Harvard
University in 1940, joining the U.S. Naval Reserve
the following year. During World War II, he
commanded PT boats in the Pacific theater. Kennedy's
survival following the 

In [44]:
from IPython.display import Markdown,display 

display(Markdown(output['output']))

John F. Kennedy's early childhood education and his education at Harvard were not similar. His early childhood education was marked by attendance at several private schools, including the Dexter School and the Riverdale Country School, where he developed an interest in history and literature. In contrast, his education at Harvard was more focused on athletics, social life, and writing, with little involvement in campus politics. While he was voted "most likely to succeed" at Choate, his preparatory boarding school, his experiences at Harvard were more geared towards extracurricular activities and social clubs, such as the Hasty Pudding Club and the Spee Club. Overall, Kennedy's early childhood education laid the foundation for his future interests and ambitions, while his education at Harvard provided him with a more well-rounded experience and opportunities for personal growth.